# Flash Attention 2: IO-Aware Exact Attention, Memory Math, and PyTorch Implementation

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/research_1)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch transformers peft bitsandbytes

In [ ]:
import torch

def online_softmax_demo(scores: torch.Tensor) -> torch.Tensor:
    """
    Demonstrate online (incremental) softmax computation.
    Processes blocks of size 2 to show the mechanics.
    """
    N = scores.shape[0]
    block_size = 2

    m = torch.tensor(float('-inf'))  # running max
    l = torch.tensor(0.0)            # running sum of exp(x - m)
    output = torch.zeros(N)

    stored_blocks = []

    for i in range(0, N, block_size):
        block = scores[i:i+block_size]
        m_new = torch.max(m, block.max())

        # Rescale previous contributions with updated max
        l_new = torch.exp(m - m_new) * l + torch.exp(block - m_new).sum()

        stored_blocks.append(block)
        m = m_new
        l = l_new

    # Final pass: compute softmax values
    for i, block in enumerate(stored_blocks):
        start = i * block_size
        output[start:start+len(block)] = torch.exp(block - m) / l

    return output

scores = torch.tensor([1.0, 3.0, 2.0, 0.5, 4.0, 1.5])
online_result = online_softmax_demo(scores)
reference = torch.softmax(scores, dim=0)

print("Online softmax: ", online_result.round(decimals=4))
print("Reference:      ", reference.round(decimals=4))
print("Max diff:       ", (online_result - reference).abs().max().item())

In [ ]:
import torch
import math

def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                             block_size: int = 64) -> torch.Tensor:
    """
    Educational implementation of Flash Attention 2 forward pass.
    Q, K, V: (batch, heads, seq_len, head_dim)
    Returns: output O of same shape as Q
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)

    O = torch.zeros_like(Q)
    L = torch.zeros(B, H, N, device=Q.device, dtype=Q.dtype)  # log-sum-exp normalizer

    # Outer loop: iterate over blocks of Q (rows)
    for q_start in range(0, N, block_size):
        q_end = min(q_start + block_size, N)
        Q_block = Q[:, :, q_start:q_end, :]  # (B, H, Bq, d)

        # Running statistics for this Q block
        m_i = torch.full((B, H, q_end - q_start), float('-inf'), device=Q.device)
        l_i = torch.zeros(B, H, q_end - q_start, device=Q.device)
        O_i = torch.zeros(B, H, q_end - q_start, d, device=Q.device)

        # Inner loop: iterate over blocks of K, V (columns)
        for kv_start in range(0, N, block_size):
            kv_end = min(kv_start + block_size, N)
            K_block = K[:, :, kv_start:kv_end, :]  # (B, H, Bkv, d)
            V_block = V[:, :, kv_start:kv_end, :]

            # Compute attention scores for this tile
            S_block = torch.einsum('bhid,bhjd->bhij', Q_block, K_block) * scale  # (B,H,Bq,Bkv)

            # Incremental softmax update
            m_block = S_block.max(dim=-1).values  # (B, H, Bq)
            m_new = torch.maximum(m_i, m_block)

            # Rescale previous output and normalization
            exp_diff = torch.exp(m_i - m_new)  # (B, H, Bq)
            O_i = O_i * exp_diff.unsqueeze(-1)
            l_i = l_i * exp_diff

            # Add new block's contribution
            P_block = torch.exp(S_block - m_new.unsqueeze(-1))  # (B,H,Bq,Bkv)
            O_i = O_i + torch.einsum('bhij,bhjd->bhid', P_block, V_block)
            l_i = l_i + P_block.sum(dim=-1)

            m_i = m_new

        # Normalize and write output
        O[:, :, q_start:q_end, :] = O_i / l_i.unsqueeze(-1)
        L[:, :, q_start:q_end] = m_i + torch.log(l_i)

    return O


# Verify against standard attention
torch.manual_seed(42)
B, H, N, d = 2, 4, 128, 64

Q = torch.randn(B, H, N, d)
K = torch.randn(B, H, N, d)
V = torch.randn(B, H, N, d)

# Flash attention output
O_flash = flash_attention_forward(Q, K, V, block_size=32)

# Standard attention output (reference)
scale = 1.0 / math.sqrt(d)
S = torch.einsum('bhid,bhjd->bhij', Q, K) * scale
P = torch.softmax(S, dim=-1)
O_ref = torch.einsum('bhij,bhjd->bhid', P, V)

max_diff = (O_flash - O_ref).abs().max().item()
print(f"Max absolute difference: {max_diff:.2e}")
print(f"Output shape: {O_flash.shape}")
print(f"Match (tol=1e-5): {max_diff < 1e-5}")

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
B, H, N, d = 2, 8, 512, 64
device = "cuda" if torch.cuda.is_available() else "cpu"

Q = torch.randn(B, H, N, d, device=device)
K = torch.randn(B, H, N, d, device=device)
V = torch.randn(B, H, N, d, device=device)

# PyTorch 2.0+ SDPA — automatically uses Flash Attention on CUDA
with torch.backends.cuda.sdp_kernel(
    enable_flash=True,
    enable_math=False,
    enable_mem_efficient=False
):
    output = F.scaled_dot_product_attention(Q, K, V)

print(f"Output shape: {output.shape}")
print(f"Device: {output.device}")

In [ ]:
from transformers import AutoModelForCausalLM

# Enable Flash Attention 2 in Hugging Face transformers
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Model loaded with Flash Attention 2")
print(f"Attention implementation: {model.config._attn_implementation}")

In [ ]:
import torch
import torch.nn.functional as F

Q = torch.randn(1, 8, 1024, 64, device="cuda" if torch.cuda.is_available() else "cpu")
K = torch.randn_like(Q)
V = torch.randn_like(Q)

# Causal masking at zero extra memory cost
output = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
print(f"Causal output shape: {output.shape}")